In [7]:
%reload_ext autoreload
%autoreload 2

In [26]:
from kret_studies import *
from kret_studies.notebook import *
from kret_studies.complex import *
from kret_studies.openai_gym import *

logger = get_notebook_logger()

/Users/Akseldkw/coding/kretsinger/data/nb_log.log


In [9]:
env_vars = os.environ.copy()
wand_db_dir = env_vars["OUTPUT_DIR"] + "/wandb"

In [10]:
# # wandb set up for logging runs online and moving them to the leaderboard
# # create a wandb account when prompted, or simply sign in if you already have an account
# # !pip install wandb -qqq
# import wandb

# wandb.login()
# run = wandb.init(dir=wand_db_dir)

In [11]:
## DO NOT CHANGE THIS CELL
import numpy as np
import gymnasium as gym
from gymnasium.envs.toy_text.frozen_lake import FrozenLakeEnv

env = typing.cast(FrozenLakeEnv, gym.make("FrozenLake-v1", is_slippery=True))
# env.seed(0)

In [21]:
T_max = 10_000

In [22]:
env.observation_space = typing.cast(gym.spaces.Discrete, env.observation_space)
env.action_space = typing.cast(gym.spaces.Discrete, env.action_space)

S, A = env.observation_space.n, env.action_space.n
theta = np.zeros((S, A))  # actor params (preferences)
V = np.zeros(S)  # critic (state values)
alpha_v, alpha_pi, gamma = 0.2, 0.1, 0.99


def softmax(prefs):  # prefs: shape (A,)
    z = np.exp(prefs - prefs.max())
    return z / z.sum()


s, info = env.reset()
for t in tqdm.tqdm(range(T_max)):
    # ---- ACTOR: sample action from πθ ----
    pi = softmax(theta[s])  # π(a|s)
    a = np.random.choice(A, p=pi)

    # ---- ENV STEP ----
    s2, r, done, _, info = env.step(a)

    # ---- CRITIC: TD error & value update ----
    delta = r + (0 if done else gamma * V[s2]) - V[s]
    V[s] += alpha_v * delta

    # ---- ACTOR: policy gradient step (using TD error as advantage) ----
    grad_logpi = -pi
    grad_logpi[a] += 1.0  # ∇θ log π(a|s) for softmax
    theta[s] += alpha_pi * delta * grad_logpi

    s, info = env.reset() if done else (s2, {})

100%|██████████| 10000/10000 [00:00<00:00, 29485.57it/s]


In [ ]:
dtt(theta, V, names=["Policy parameters (theta)", "State values (V)"])

,0,1,2,3
0,0.109153,-0.016520,0.003710,-0.096343
1,-0.105672,-0.019848,-0.009939,0.135459
2,0.085373,-0.005171,0.031569,-0.111771
3,0.000401,-0.006269,-0.022797,0.028665
4,0.199869,0.087552,-0.043281,-0.244140
5,0.000000,0.000000,0.000000,0.000000
6,0.063516,0.005265,0.123478,-0.192259
7,0.000000,0.000000,0.000000,0.000000
8,-0.281243,0.064190,-0.007063,0.224116
9,-0.086512,0.178221,0.011142,-0.102850
